# Get from LHE

Reads the LHE files in order to get the known valid points, getting the the list of important parameters

In [4]:
import os
import glob
import re
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR = "../../valid_points_lhe"    
OUTPUT_CSV = "valid_points_extended.csv"

# ─────────────────────────────────────────────────────────────────────────────
# 1. HERRAMIENTAS DE PARSEO
# ─────────────────────────────────────────────────────────────────────────────
# Regex Robusto: Acepta enteros, flotantes y notación científica
line_re = re.compile(r'^\s*(\d+)\s+([+-]?\d+(?:\.\d*)?(?:[eE][+-]?\d+)?)\s*(?:#.*)?$')

# Mapa de Partículas para Nombres de BR (PDG ID -> Nombre legible)
PDG_MAP = {
    1: "d",  2: "u",  3: "s",  4: "c",  5: "b",  6: "t",
    11: "e", 12: "nue", 13: "mu", 14: "numu", 15: "ta", 16: "nuta",
    21: "g", 22: "ga", 23: "Z", 24: "W",
    25: "h1", 35: "h2", 36: "A", 37: "Hp"
}

def get_particle_name(pid):
    """Devuelve nombre limpio (ej: 5 -> 'b', -5 -> 'b')"""
    pid = abs(int(pid))
    return PDG_MAP.get(pid, str(pid))

# ─────────────────────────────────────────────────────────────────────────────
# 2. DEFINICIÓN DE BLOQUES DE INTERÉS
# ─────────────────────────────────────────────────────────────────────────────
MINPAR_MAP = {
    3:  "tan_beta",
    11: "lambda1", 12: "lambda2", 13: "lambda3", 14: "lambda4",
    15: "lambda5", 16: "lambda6", 17: "lambda7",
    18: "m12_2",   20: "sin_ba",  21: "cos_ba",  24: "yukawa_type",
}
MASS_MAP = {
    25: "Mh1", 35: "Mh2", 36: "Mh3", 37: "MHp"
}
THDM_MAP = {
    1: "valid_param", 2: "unitarity_ok", 3: "perturbativity_ok", 4: "stability_ok"
}

# ─────────────────────────────────────────────────────────────────────────────
# 3. MOTOR DE EXTRACCIÓN
# ─────────────────────────────────────────────────────────────────────────────
records = []
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.lha")))
print(f"🚀 Procesando {len(files)} archivos LHA...")

for path in files:
    rec = {"file": os.path.basename(path)}
    block = None
    
    with open(path, 'r') as f:
        for line in f:
            line_clean = line.strip()
            if not line_clean or line_clean.startswith('#'):
                continue
            
            up = line_clean.upper()

            # --- Detección de Bloques ---
            if up.startswith("BLOCK MINPAR"):
                block = "MINPAR"
                continue
            elif up.startswith("BLOCK MASS"):
                block = "MASS"
                continue
            elif up.startswith("BLOCK THDM"):
                block = "THDM"
                continue
            elif up.startswith("DECAY"):
                # Detectar si es el Higgs pesado (ID 35)
                parts = up.split()
                if len(parts) >= 2 and parts[1] == "35":
                    block = "DECAY_35"
                    # Opcional: Extraer ancho total de la cabecera "DECAY 35 1.23E-03"
                    if len(parts) >= 3:
                        try: rec["total_width_h2"] = float(parts[2])
                        except: pass
                else:
                    block = None # Otro decay que no nos interesa
                continue
            elif up.startswith("BLOCK"):
                block = None
                continue

            # --- Extracción de Datos ---
            
            # CASO A: Parámetros y Constraints (MINPAR, MASS, THDM)
            if block in ("MINPAR", "MASS", "THDM"):
                m = line_re.match(line)
                if m:
                    code = int(m.group(1))
                    val  = float(m.group(2))
                    
                    if block == "MINPAR" and code in MINPAR_MAP:
                        rec[MINPAR_MAP[code]] = val
                    elif block == "MASS" and code in MASS_MAP:
                        rec[MASS_MAP[code]] = val
                    elif block == "THDM" and code in THDM_MAP:
                        rec[THDM_MAP[code]] = int(val) # Flag booleano

            # CASO B: Branching Ratios (DECAY_35)
            elif block == "DECAY_35":
                # Formato: BR  NDA  ID1  ID2 ...
                parts = line_clean.split()
                try:
                    # Validar que parece una línea de BR (al menos 4 columnas numéricas al inicio)
                    # Ojo: A veces hay comentarios al final
                    br = float(parts[0])
                    # nda = parts[1]
                    id1 = parts[2]
                    id2 = parts[3]
                    
                    name1 = get_particle_name(id1)
                    name2 = get_particle_name(id2)
                    
                    # Ordenar nombres para consistencia (ga_Z es lo mismo que Z_ga)
                    channel = "_".join(sorted([name1, name2]))
                    col_name = f"BR_h2_{channel}"
                    
                    rec[col_name] = br
                except (ValueError, IndexError):
                    continue

    records.append(rec)

# ─────────────────────────────────────────────────────────────────────────────
# 4. LIMPIEZA Y GUARDADO
# ─────────────────────────────────────────────────────────────────────────────
df = pd.DataFrame(records)

# A. Filtrado de Higgs SM (125 GeV)
# Tolerancia típica de ±1 GeV para simulaciones LHE
if 'Mh1' in df.columns:
    initial_len = len(df)
    df = df[ (df['Mh1'] > 124.0) & (df['Mh1'] < 126.0) ]
    print(f"✂️  Filtro Higgs (125±1 GeV): {initial_len} -> {len(df)} puntos.")
else:
    print("⚠️  Advertencia: Columna 'Mh1' no encontrada. No se aplicó filtro de Higgs.")

# B. Organización de Columnas
base_cols = ["file"] + list(MINPAR_MAP.values()) + list(MASS_MAP.values()) + list(THDM_MAP.values()) + ["total_width_h2"]
# Identificar columnas BR que aparecieron dinámicamente
br_cols = sorted([c for c in df.columns if c.startswith("BR_h2_")])
# Columnas finales ordenadas
final_cols = [c for c in base_cols if c in df.columns] + br_cols

df = df.reindex(columns=final_cols)

# C. Rellenar ceros en BRs (Si un canal no aparece, su BR es 0)
df[br_cols] = df[br_cols].fillna(0.0)

# Guardar
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Guardado exitoso: {OUTPUT_CSV} ({len(df)} registros)")
print(df.head())

🚀 Procesando 23 archivos LHA...
✂️  Filtro Higgs (125±1 GeV): 23 -> 18 puntos.
✅ Guardado exitoso: valid_points_extended.csv (18 registros)
      file  tan_beta   lambda1   lambda2   lambda3   lambda4   lambda5  \
3  130.lha   10000.0  1.020619  0.257734  2.669307 -1.205802 -1.205802   
4  140.lha   10000.0  0.976086  0.257734  2.580235 -1.161265 -1.161265   
5  150.lha   10000.0  0.928255  0.257734  2.484564 -1.113430 -1.113430   
6  160.lha   10000.0  0.877126  0.257734  2.382295 -1.062296 -1.062296   
7  170.lha   10000.0  0.822698  0.257734  2.273428 -1.007862 -1.007862   

   lambda6  lambda7     m12_2  ...  BR_h2_Z_ga  BR_h2_b_b  BR_h2_c_c  \
3      0.1      0.0  1.689909  ...    0.055195   0.392524   0.017606   
4      0.1      0.0  1.959909  ...    0.084123   0.346920   0.015545   
5      0.1      0.0  2.249909  ...    0.114097   0.305774   0.013690   
6      0.1      0.0  2.559909  ...    0.143431   0.269362   0.012052   
7      0.1      0.0  2.889909  ...    0.171143   0.2375

In [5]:
df.columns

Index(['file', 'tan_beta', 'lambda1', 'lambda2', 'lambda3', 'lambda4',
       'lambda5', 'lambda6', 'lambda7', 'm12_2', 'sin_ba', 'cos_ba',
       'yukawa_type', 'Mh1', 'Mh2', 'Mh3', 'MHp', 'valid_param',
       'unitarity_ok', 'perturbativity_ok', 'stability_ok', 'total_width_h2',
       'BR_h2_W_W', 'BR_h2_Z_Z', 'BR_h2_Z_ga', 'BR_h2_b_b', 'BR_h2_c_c',
       'BR_h2_e_e', 'BR_h2_g_g', 'BR_h2_ga_ga', 'BR_h2_mu_mu', 'BR_h2_s_s',
       'BR_h2_t_t', 'BR_h2_ta_ta'],
      dtype='object')